<a href="https://colab.research.google.com/github/Koks-creator/LeafsAnalyzeWithYoloSegmentation/blob/main/yolov11_potato_leafs_segm_giten.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YOLO11 — Instance Segmentation


## 1. Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
!ln -s /content/gdrive/My\ Drive/ /mydrive

## 2. Setup and imports

In [ ]:
!pip install -q ultralytics==8.4.19 sahi==0.11.34 roboflow

In [ ]:
!pip freeze | grep -e ultralytics -e sahi -e torch -e roboflow

In [ ]:
import os, glob, shutil, yaml, random
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from ultralytics import YOLO
from roboflow import Roboflow

In [ ]:
MODEL_DIR = "./models"
MODEL_NAME = "yolo11m-seg.pt"     # n/s/m/l/x-seg
MODEL_PATH = os.path.join(MODEL_DIR, MODEL_NAME)

IMGSZ = 640
BATCH_SIZE = 8 # -1 = autobatch
WORKERS = 8
EPOCHS = 150
RUN_NAME = "yolo11m_seg_leafs"
PROJECT = "/mydrive/yoloTrain/runs"

# 3. Downloading dataset

Dataset must be **Instance Segmentation** type


In [ ]:
use_local = False
resume = False

In [ ]:
CLASS_NAMES = [
    "your classes",
]

if use_local:
    DATASET_PATH  = "/content/train_data"
    CONFIG_YAML_FILE = "./config/data.yaml"

    !unzip -q /mydrive/yoloTrain/train_data.zip -d /content

    data_yaml = {
        'path' : DATASET_PATH,
        'train': 'images/train',
        'val'  : 'images/val',
        'test' : 'images/test',
        'names': {i: n for i, n in enumerate(CLASS_NAMES)},
    }

    os.makedirs("config", exist_ok=True)
    with open(CONFIG_YAML_FILE, 'w') as f:
        yaml.dump(data_yaml, f, sort_keys=False, allow_unicode=True)

    print("Config:", CONFIG_YAML_FILE)

else:

    rf = Roboflow(api_key="")
    try:
        project = rf.workspace("potato-leaf-disease-in-uncontrolled-environment") \
                    .project("potato-leaf-disease-nzxek")
    except Exception as e:
        print("Original one doesn't work, fok that punk, let's take the fork", e)
        project = rf.workspace("potato-64bwf").project("potato-leaf-disease-nzxek-otyen")

    # rf = Roboflow(api_key="")
    # project = rf.workspace("agrobotanix").project("potato-leaf-m4yzg")

    ver = max(v.version if isinstance(v.version, int) else int(str(v.version).split("/")[-1])
              for v in project.versions())
    dataset = project.version(ver).download("yolov11", location="ds/leaf")

    ver = max(v.version if isinstance(v.version, int) else int(str(v.version).split("/")[-1])
              for v in project.versions())
    dataset = project.version(ver).download("yolov11")

    DATASET_PATH = dataset.location
    CONFIG_YAML_FILE = os.path.join(DATASET_PATH, "data.yaml")
    print(open(CONFIG_YAML_FILE).read())


shit for agrobotanix dataset, to get only whole leafs example, get rif off examples with just diseased area

In [ ]:
# import yaml, collections
# from pathlib import Path

# cfg = yaml.safe_load(open(CONFIG_YAML_FILE))
# print(cfg["names"])

# cnt = collections.Counter()
# for f in Path(DATASET_PATH).rglob("labels/*.txt"):
#     for l in f.read_text().splitlines():
#         if l.strip():
#             cnt[int(l.split()[0])] += 1

# for cid, n in sorted(cnt.items()):
#     print(cid, cfg["names"][cid], n)

In [ ]:
# KEEP = 0   # healthy = caly lisc

# for f in Path(DATASET_PATH).rglob("labels/*.txt"):
#     keep = ["0 " + " ".join(l.split()[1:])
#             for l in f.read_text().splitlines()
#             if l.strip() and int(l.split()[0]) == KEEP]
#     f.write_text("\n".join(keep))

### 4a. Sanity-check of labels



In [ ]:
def check_labels(root, split="train", n_show=3):
    lbl_dir = os.path.join(root, split, "labels")
    if not os.path.isdir(lbl_dir):
        lbl_dir = os.path.join(root, "labels", split)
    files = sorted(glob.glob(os.path.join(lbl_dir, "*.txt")))
    assert files, f"no labels i chuj in {lbl_dir}"

    tok_counts, n_inst, empties = [], 0, 0
    for f in files:
        lines = [l for l in open(f).read().splitlines() if l.strip()]
        if not lines:
            empties += 1
        for l in lines:
            tok_counts.append(len(l.split()))
            n_inst += 1

    tok = np.array(tok_counts)
    print(f"[{split}] files: {len(files)} | instances: {n_inst} | empty files: {empties}")
    print(f"[{split}] tokens in line: min={tok.min()} median={int(np.median(tok))} max={tok.max()}")

    if (tok == 5).all(): # class id, x1, y2, w, h
        print("5 TOKEN, IT'S REGULAR DETECTION SHIT")
    elif (tok >= 7).all() and (tok % 2 == 1).all():
        print("OK")
    else:
        print("SOME WEIRD, MIX SHIT")

    print("Let's see:")
    for f in files[:n_show]:
        line = open(f).readline().strip()
        print(f"  {os.path.basename(f)}: {line[:110]}{'...' if len(line) > 110 else ''}")

check_labels(DATASET_PATH, "train")

### 4b. Preview


In [ ]:
def preview_gt(root, split="train", n=10, seed=0):
    img_dir = os.path.join(root, split, "images")
    if not os.path.isdir(img_dir):
        img_dir = os.path.join(root, "images", split)
    lbl_dir = img_dir.replace("images", "labels")

    imgs = sorted(glob.glob(os.path.join(img_dir, "*")))
    random.Random(seed).shuffle(imgs)
    imgs = imgs[:n]

    fig, axes = plt.subplots(1, len(imgs), figsize=(6 * len(imgs), 6))
    axes = np.atleast_1d(axes)

    for ax, ip in zip(axes, imgs):
        img = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        lp = os.path.join(lbl_dir, Path(ip).stem + ".txt")

        overlay = img.copy()
        if os.path.exists(lp):
            for line in open(lp).read().splitlines():
                p = line.split()
                if len(p) < 7:
                    continue
                cls = int(p[0])
                pts = np.array(p[1:], dtype=float).reshape(-1, 2) * [w, h]
                color = [(255, 60, 60), (60, 255, 120), (80, 160, 255),
                         (255, 200, 60), (200, 80, 255)][cls % 5]
                cv2.fillPoly(overlay, [pts.astype(np.int32)], color)
                cv2.polylines(img, [pts.astype(np.int32)], True, color, 2)

        ax.imshow(cv2.addWeighted(overlay, 0.4, img, 0.6, 0))
        ax.set_title(Path(ip).name, fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

preview_gt(DATASET_PATH, "train")

## 5. GPU

In [ ]:
if torch.cuda.is_available():
    device = 0
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = "cpu"
    print("Damn, no GPU — CPU.")

## 6. Model

In [ ]:
os.makedirs(MODEL_DIR, exist_ok=True)

if resume:
    MODEL_PATH = f"{PROJECT}/{RUN_NAME}/weights/last.pt"
elif not os.path.exists(MODEL_PATH):
    YOLO(MODEL_NAME)
    if os.path.exists(MODEL_NAME):
        shutil.move(MODEL_NAME, MODEL_PATH)

model = YOLO(MODEL_PATH)
print("Wagi:", MODEL_PATH)
print("task:", model.task)
assert model.task == "segment", "No segment weights!!11!!11"

## 7. Training

In [ ]:
if not resume:
    results = model.train(
        data=CONFIG_YAML_FILE,
        epochs=EPOCHS,
        patience=60,
        imgsz=IMGSZ,
        batch=BATCH_SIZE,
        cache="ram",
        overlap_mask=False,
        mask_ratio=4,
        copy_paste=0.5,
        copy_paste_mode="flip",
        optimizer="AdamW",
        lr0=0.001,
        lrf=0.01,
        cos_lr=True,
        warmup_epochs=5,
        weight_decay=0.0005,
        degrees=45.0,
        scale=0.5,
        shear=2.0,
        perspective=0.0005,
        translate=0.1,
        fliplr=0.5,
        flipud=0.5,
        mosaic=1.0,
        close_mosaic=20,
        mixup=0.0,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        box=7.5,
        cls=0.5,
        dfl=1.5,
        seed=0,
        plots=True,
        project=PROJECT,
        name=RUN_NAME,
        save=True, save_period=25,
        device=device, workers=WORKERS,
        amp=True, verbose=True,
    )
else:
    results = model.train(resume=True)

In [ ]:
BEST = f"{PROJECT}/{RUN_NAME}/weights/best.pt"
print(BEST, "->", os.path.exists(BEST))

## 8. Validation

The segmentation model returns **two sets of metrics**. `metrics.box.*` describes the quality
of the boxes, and in the `-seg` model, it still exists—it’s easy to mistakenly report it
as a segmentation result. What matters is `metrics.seg.*`.

A large discrepancy between them is diagnostic: a high `box mAP` with a low
`seg mAP` means that the model finds objects but outlines them poorly—usually due to too low
a mask resolution (`mask_ratio` / `imgsz`).

Translated with DeepL.com (free version)

In [ ]:
model = YOLO(BEST)
metrics = model.val(data=CONFIG_YAML_FILE, imgsz=768, iou=0.85)

In [ ]:
model = YOLO(BEST)
metrics = model.val(
    data=CONFIG_YAML_FILE,
    split="test",
    imgsz=IMGSZ,
    conf=0.001,
    iou=0.6,
    plots=True,
)

print(f"BOX   mAP50-95: {metrics.box.map:.4f} | mAP50: {metrics.box.map50:.4f}")
print(f"MASK  mAP50-95: {metrics.seg.map:.4f} | mAP50: {metrics.seg.map50:.4f}")
print(f"delta (box - mask) mAP50-95: {metrics.box.map - metrics.seg.map:+.4f}")

In [ ]:
print(f"BOX   mAP50-95: {metrics.box.map:.4f} | mAP50: {metrics.box.map50:.4f}")
print(f"MASK  mAP50-95: {metrics.seg.map:.4f} | mAP50: {metrics.seg.map50:.4f}")
print(f"delta (box - mask) mAP50-95: {metrics.box.map - metrics.seg.map:+.4f}")

In [ ]:
rows = []
for i, c in enumerate(metrics.seg.ap_class_index):
    rows.append({
        "class"    : model.names[c],
        "P_mask"   : round(float(metrics.seg.p[i]), 4),
        "R_mask"   : round(float(metrics.seg.r[i]), 4),
        "AP50_mask": round(float(metrics.seg.ap50[i]), 4),
        "AP50_box" : round(float(metrics.box.ap50[i]), 4),
    })

pd.DataFrame(rows).sort_values("AP50_mask")

## 9. Prediction

`retina_masks=True` - that's important, without it masks are in model's input resolution instead of original img resolution.

In [ ]:
conf = 0.4
iou_nms = 0.7

test_imgs = sorted(glob.glob(os.path.join(DATASET_PATH, "test", "images", "*")))
if not test_imgs:
    test_imgs = sorted(glob.glob(os.path.join(DATASET_PATH, "valid", "images", "*")))
print(f"Zdjęć testowych: {len(test_imgs)}")

res = model.predict(test_imgs[0], retina_masks=True, conf=conf, iou=iou_nms, verbose=False)[0]

print("orig_shape :", res.orig_shape)
print("maski      :", tuple(res.masks.data.shape) if res.masks is not None else "brak")
print("instancji  :", 0 if res.masks is None else len(res.masks))

plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB))
plt.axis("off"); plt.show()

## 10. Area Measurement

This is where the fun begins, here we're actually calcuating area

I calculate `area_pct` based on the dimensions of the mask itself, not the original—this ensures that the result
is correct regardless of whether `retina_masks` worked. I convert `area_px`
back to the original scale.

**px→mm calibration:** You need an object of known dimensions in the frame (a ruler,
a coin, a calibration marker). Measure its length in pixels and divide it by
its actual length. The calibration is only valid for a fixed acquisition geometry—
changing the camera distance invalidates it.

In [ ]:
PX_PER_MM = None      # e.g., 12.5  (pixels per millimeter); None = pixels and % only

def measure(results_iter, names, px_per_mm=None):
    rows = []
    for r in results_iter:
        h, w = r.orig_shape
        img  = Path(r.path).name

        if r.masks is None or len(r.masks) == 0:
            rows.append({"image": img, "inst_id": None, "class": None, "conf": None,
                         "area_pct": 0.0, "area_px": 0, "area_mm2": None})
            continue

        m = r.masks.data.bool().cpu()
        mh, mw = m.shape[-2:]
        denom = mh * mw

        for i in range(len(m)):
            pct = 100.0 * m[i].sum().item() / denom
            px  = int(round(pct / 100.0 * h * w))
            rows.append({
                "image"   : img,
                "inst_id" : i,
                "class"   : names[int(r.boxes.cls[i])],
                "conf"    : round(float(r.boxes.conf[i]), 3),
                "area_pct": round(pct, 4),
                "area_px" : px,
                "area_mm2": round(px / px_per_mm**2, 3) if px_per_mm else None,
            })
    return pd.DataFrame(rows)

stream = model.predict(source=test_imgs, stream=True, retina_masks=True,
                       conf=CONF, iou=IOU_NMS, verbose=False)

df = measure(stream, model.names, PX_PER_MM)
df.head(20)

## Cheat Sheet — Segmentation-Specific Arguments

| Argument | Default | Range | When to Use |
|---|---|---|---|
| `overlap_mask` | `True` | bool | `False` only for transparent, overlapping objects |
| `mask_ratio` | `4` | 1–8 | `2` for thin structures; `8` when speed matters and objects are large |
| `copy_paste` | `0.0` | 0–1 | 0.2–0.4 for sparse classes; 0.5–0.7 for severe class imbalance |
| `copy_paste_mode` | `flip` | flip / mixup | `mixup` when visual diversity is more important than position |
| `retina_masks` *(predict)* | `False` | bool | `True` whenever you measure area |

**Post-training diagnostics**

| Symptom | Probable cause |
|---|---|
| `seg mAP` ≈ `box mAP`, rectangular masks | you’re training on detection labels |
| `box mAP` high, `seg mAP` low | mask resolution too low — `mask_ratio=2` or larger `imgsz` |
| jagged mask edges | same as above + consider using `yolo11m-seg` instead of `s` |
| good `val`, poor on production | augmentation does not cover real-world acquisition conditions |
| CUDA OOM with the same `imgsz` as detection | normal — reduce the batch size or set `batch=-1` |